# AutoPath trên Kaggle
1. **Settings → Accelerator → GPU T4 x2 hoặc P100**, bật **Internet**.
2. **Add Input → Datasets →** tìm `tusimple` (manideep1108/tusimple).
3. Đưa code lên: tạo Kaggle Dataset từ file `AutoPath.zip` (tên `autopath-code`) **hoặc** đẩy lên GitHub rồi clone ở ô dưới.
4. Chạy lần lượt từng ô. Mỗi lần train khoảng 1–2 giờ, nên chạy bằng **Save Version (Run All)** để không bị ngắt giữa chừng.

In [ ]:
import os, glob, shutil
WORK = '/kaggle/working'
os.chdir(WORK)
# Cách A: code upload dưới dạng Kaggle Dataset
src = glob.glob('/kaggle/input/**/AutoPath/train.py', recursive=True)
if src and not os.path.exists('AutoPath'):
    shutil.copytree(os.path.dirname(src[0]), 'AutoPath')
# Cách B: clone từ GitHub của nhóm (sửa URL)
# !git clone https://github.com/<tai-khoan>/AutoPath.git
os.chdir(f'{WORK}/AutoPath')
!pip -q install albumentations gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Tìm thư mục TuSimple
hits = glob.glob('/kaggle/input/**/label_data_0313.json', recursive=True)
TUS = os.path.dirname(os.path.dirname(hits[0])) if hits else None
print('TUS =', TUS)
print(glob.glob(f'{TUS}/**/test_label*.json', recursive=True))

In [ ]:
# Kiểm tra nhãn ego-path (lần đầu chạy sẽ đọc JSON và lưu cache)
from autopath.data import load_or_build
for s in ['train', 'val', 'test']:
    d = load_or_build(TUS, s, 'cache')
    print(s, len(d['records']), d['stats'])

In [ ]:
# Xem thử vài nhãn
import cv2, matplotlib.pyplot as plt
from autopath.data import crop_resize
from autopath.smoothing import draw_path
d = load_or_build(TUS, 'train', 'cache')['records']
fig, ax = plt.subplots(2, 3, figsize=(15, 5))
for a, r in zip(ax.ravel(), d[::len(d)//6]):
    im = draw_path(crop_resize(cv2.imread(r['image']), 1024, 512), r['xp'], r['valid'])
    a.imshow(im[..., ::-1]); a.axis('off')
plt.tight_layout(); plt.savefig('figures_label_samples.png', dpi=150)

## Thí nghiệm 1 + 2: huấn luyện

In [ ]:
!python train.py --data-root $TUS --backbone resnet18 --name res18_aug --workers 4

In [ ]:
!python train.py --data-root $TUS --backbone mobilenet_v3_small --name mbv3s_aug --workers 4

In [ ]:
!python train.py --data-root $TUS --backbone resnet18 --no-aug --name res18_noaug --workers 4

In [ ]:
# (tùy chọn) đầu GAP như đề cương ban đầu
# !python train.py --data-root $TUS --backbone resnet18 --head gap --name res18_gap --workers 4

## Đánh giá trên tập test TuSimple

In [ ]:
for r in ['res18_aug', 'mbv3s_aug', 'res18_noaug']:
    !python evaluate.py --data-root $TUS --method autopath --ckpt runs/{r}/best.pt --out results/{r}
!python evaluate.py --data-root $TUS --method classical --out results/classical

## Thí nghiệm 3: AutoSteer 2.0
Lưu ý: AutoSteer đã dùng khoảng 75% tập test TuSimple để huấn luyện (xem README), nên kết quả trên tập này chỉ để tham khảo. So sánh chính được làm trên tập TP.HCM.

In [ ]:
!git clone -q https://github.com/autowarefoundation/auto_steer.git
os.makedirs('weights', exist_ok=True)
!gdown -q https://drive.google.com/uc?id=1-wfPyu7HId7YDSh_T_DkB3Ma1zPc7Fu_ -O weights/autosteer.pth
!python evaluate.py --data-root $TUS --method autosteer --autosteer-repo auto_steer --autosteer-weights weights/autosteer.pth --out results/autosteer

In [ ]:
# Tập TP.HCM (upload thư mục hcmc gồm frames/ và labels.json thành Kaggle Dataset)
H = glob.glob('/kaggle/input/**/hcmc/labels.json', recursive=True)
if H:
    J = H[0]
    !python evaluate.py --custom-json {J} --method autopath --ckpt runs/res18_aug/best.pt --out results/res18_hcmc
    !python evaluate.py --custom-json {J} --method autopath --ckpt runs/mbv3s_aug/best.pt --out results/mbv3s_hcmc
    !python evaluate.py --custom-json {J} --method autosteer --autosteer-repo auto_steer --autosteer-weights weights/autosteer.pth --out results/autosteer_hcmc
    !python evaluate.py --custom-json {J} --method classical --out results/classical_hcmc

## Tốc độ
FPS trên GPU Kaggle chỉ để tham khảo. **Số FPS ghi vào báo cáo phải đo trên RTX 3050 của nhóm**: tải checkpoint về rồi chạy `benchmark_fps.py` ở máy.

In [ ]:
!python benchmark_fps.py --ckpts runs/res18_aug/best.pt runs/mbv3s_aug/best.pt --autosteer-repo auto_steer --autosteer-weights weights/autosteer.pth --classical --out results/benchmark_kaggle.json

## Hình, bảng và đóng gói kết quả

In [ ]:
!python tools/make_figures.py --runs "runs/*" --results "results/*/results.json" --benchmark results/benchmark_kaggle.json --out figures
print(open('figures/tables.md', encoding='utf-8').read())
!zip -qr /kaggle/working/autopath_outputs.zip runs/*/best.pt runs/*/history.csv runs/*/config.json results figures